# Evaluation of ERA5 precipitation estimates

This notebooks demonstrates how to use the ``ipwgml.evaluation.Evaluator`` class to evaluate GPROF V7 retrievals. Since the evaluation of GPROF retrievals is largely similar to the evaluation of the IMERG retrievals, this example will not explain all the configuration details. For a more thorough introduction to using the ``Evaluator`` class, refer to [the IMERG example](evaluat_imerge.ipynb).

In [1]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
from typing import Tuple

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

## The retrieval callback

With the helper functions in place, it is easy to to implement a retrieval callback function. The function extracts the granule number and scan range from the attributes of the ``input_data`` dataset and then uses the two helper functions above to load the corresponding data from the GPROF file. Note that the evaluator will take care of mapping the swath-based estimates from GPROF to the regular latitude-longitude grid used for the SPR dataset.

In [2]:
from satrain.input import Ancillary
anc = Ancillary(variables=["total_precipitation"])

def retrieve_era5(input_data: xr.Dataset) -> xr.Dataset:
    """
    Retrieval callback function to load GPROF data corresponding to IPWGML SPR evaluation data.

    Args:
        input_data: An xarray.Dataset containing the retrieval input data.

    Return:
        An xarray.Dataset containing the retrieval results.
    """
    lons = input_data.longitude.data
    lats = input_data.latitude.data
    tp = 1e3 * input_data.ancillary[0].data
    precip_flag = 0.1 < tp
    heavy_precip_flag = 10 < tp
        
    return xr.Dataset({
        "surface_precip": (("latitude", "longitude"), tp),
        "precip_flag": (("latitude", "longitude"), precip_flag),
        "heavy_precip_flag": (("latitude", "longitude"), heavy_precip_flag),
    })

# Evaluating GPROF retrievals

For the evaluation of the GPROF retrievals we instantiate the evaluator with the ``on_swath`` geometry because the GPROF V7 results are provided at the nominal GMI footprint positions. The evaluator will then automatically map these results to the nearest gridded MRMS measurements.

In [3]:
from pathlib import Path

from satrain.evaluation import Evaluator
from satrain.target import TargetConfig

## Case study

To verify the implementation of the ``retrieval_fn`` callback, we display the results for a single scene using the evaluator's ``plot_retrieval_results`` function.

## Running the evaluator

We use the ``n_processes`` argument to run the evaluation using multiple processes in parallel.

In [ ]:
for sensor in ["gmi", "atms"]:
    for domain in ["conus", "korea", "austria"]:
        evaluator = Evaluator(
            domain=domain,
            base_sensor=sensor,
            geometry="gridded",
            retrieval_input=[anc],
            download=True,
        )
        evaluator.evaluate(retrieval_fn=retrieve_era5)
        results = evaluator.get_results()
        results.to_netcdf(f"era5_{sensor}_{domain}.nc")

Output()